In [8]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import PCA

df = pd.read_csv(r'D:\Ai_projects\sentiment\data\twitter_training.csv')
df.head()

,2401,Borderlands,Positive,"im getting on borderlands and i will murder you all ,"
0,2401,Borderlands,Positive,I am coming to the borders and I will kill you...
1,2401,Borderlands,Positive,im getting on borderlands and i will kill you ...
2,2401,Borderlands,Positive,im coming on borderlands and i will murder you...
3,2401,Borderlands,Positive,im getting on borderlands 2 and i will murder ...
4,2401,Borderlands,Positive,im getting into borderlands and i can murder y...


In [ ]:
# checking for null values
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 74681 entries, 0 to 74680
Data columns (total 4 columns):
 #   Column                                                 Non-Null Count  Dtype 
---  ------                                                 --------------  ----- 
 0   2401                                                   74681 non-null  int64 
 1   Borderlands                                            74681 non-null  object
 2   Positive                                               74681 non-null  object
 3   im getting on borderlands and i will murder you all ,  73995 non-null  object
dtypes: int64(1), object(3)
memory usage: 2.3+ MB


In [ ]:
# cheking the shape of the data
df.shape
df.columns

Index(['2401', 'Borderlands', 'Positive',
       'im getting on borderlands and i will murder you all ,'],
      dtype='object')

In [4]:
#changing the column names
df.columns = ['index', 'labels', 'sentiment', 'sentence']
df.head()

,index,labels,sentiment,sentence
0,2401,Borderlands,Positive,I am coming to the borders and I will kill you...
1,2401,Borderlands,Positive,im getting on borderlands and i will kill you ...
2,2401,Borderlands,Positive,im coming on borderlands and i will murder you...
3,2401,Borderlands,Positive,im getting on borderlands 2 and i will murder ...
4,2401,Borderlands,Positive,im getting into borderlands and i can murder y...


In [5]:
#dropping the unwanted columns
df = df.drop(columns=['labels'])
df.head()

,index,sentiment,sentence
0,2401,Positive,I am coming to the borders and I will kill you...
1,2401,Positive,im getting on borderlands and i will kill you ...
2,2401,Positive,im coming on borderlands and i will murder you...
3,2401,Positive,im getting on borderlands 2 and i will murder ...
4,2401,Positive,im getting into borderlands and i can murder y...


In [6]:
#checking sentiment distribution
df['sentiment'].value_counts()

sentiment
Negative      22542
Positive      20831
Neutral       18318
Irrelevant    12990
Name: count, dtype: int64

In [7]:
# cleaning and preprocessing the text data
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer, SnowballStemmer
import re

# Init tools
stemmer = SnowballStemmer("english")
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r"http\S+|www\S+|https\S+", '', text)  # remove URLs
    text = re.sub(r'\@\w+|\#','', text)  # remove @mentions and hashtags
    text = re.sub(r'[^A-Za-z\s]', '', text)  # remove punctuation & numbers
    text = re.sub(r"http\S+|@\S+|#[A-Za-z0-9_]+", "", text)  # remove URLs, mentions, hashtags
    words = word_tokenize(text)
    words = [stemmer.stem(word) for word in words if word.isalpha() and word not in stop_words]
    words = [lemmatizer.lemmatize(word) for word in words]
    return ' '.join(words)

# Vectorized operation
df['cleaned'] = df['sentence'].apply(clean_text)


In [8]:
# checking the cleaned text
df.head()

,index,sentiment,sentence,cleaned
0,2401,Positive,I am coming to the borders and I will kill you...,come border kill
1,2401,Positive,im getting on borderlands and i will kill you ...,im get borderland kill
2,2401,Positive,im coming on borderlands and i will murder you...,im come borderland murder
3,2401,Positive,im getting on borderlands 2 and i will murder ...,im get borderland murder
4,2401,Positive,im getting into borderlands and i can murder y...,im get borderland murder


In [9]:
#vectorizing the text data using TF-IDF
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(max_features=5000,ngram_range=(1,2))  # limit to top 5000 words
X_tfidf = vectorizer.fit_transform(df['cleaned'])
print(X_tfidf.shape)

(74681, 5000)


In [ ]:
#label encoding the target variable
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()
df['label_encoded'] = label_encoder.fit_transform(df['sentiment'])  # assuming 'sentiment' column exists
df.head()

,index,sentiment,sentence,cleaned,label_encoded
74661,9197,Neutral,Nvidia therefore doesn ’ t want to give up its...,nvidia therefor want give crypto craze doc max...,2
74662,9197,Neutral,is doesn’t should I give up its password ‘cryp...,doesnt give password crypto wallet doc maxbitc...,2
74663,9198,Negative,Nvidia really delayed the 3070 2 weeks .,nvidia realli delay week,1
74664,9198,Negative,Nvidia really delayed the 3070 by 2 weeks.,nvidia realli delay week,1
74665,9198,Negative,Nvidia did delay by 3070 2 weeks.,nvidia delay week,1
74666,9198,Negative,Nvidia really delayed the 3070 several weeks.,nvidia realli delay sever week,1
74667,9198,Negative,Nvidia really only delayed the 3070 2 flight w...,nvidia realli delay flight week,1
74668,9198,Negative,Nvidia really delayed the next 2 weeks.,nvidia realli delay next week,1
74669,9199,Positive,Let no elim go unnoticed. . . . NVIDIA Highlig...,let elim go unnot nvidia highlight automat rec...,3
74670,9199,Positive,t let Elim go unnoticed.... NVIDIA Highlights ...,let elim go unnot nvidia highlight automat rec...,3


In [12]:
#splitting the data into training and testing sets
from sklearn.model_selection import train_test_split

X = X_tfidf  
y = df['label_encoded'] 
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [13]:
# evaluating the model using Logistic regression
# checking accuracy score, classification report and confusion matrix
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix


model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))


Accuracy: 0.6645243355426123

Classification Report:
               precision    recall  f1-score   support

           0       0.65      0.48      0.55      2661
           1       0.71      0.75      0.73      4471
           2       0.58      0.63      0.61      3551
           3       0.69      0.71      0.70      4254

    accuracy                           0.66     14937
   macro avg       0.66      0.65      0.65     14937
weighted avg       0.66      0.66      0.66     14937


Confusion Matrix:
 [[1283  458  442  478]
 [ 187 3369  568  347]
 [ 268  518 2247  518]
 [ 228  413  586 3027]]


Accuracy is very low. So now can we use another model for improving accuracy

In [14]:
from sklearn.ensemble import RandomForestClassifier
model = RandomForestClassifier(n_estimators=100)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))

Accuracy: 0.867309366003883


Now accuracy is good. We can check out our model with some example.

In [21]:
# EXAMPLE
new_text = "product is waste of money, not worth it"

# Clean and transform
processed_text = clean_text(new_text)
vectorized_text = vectorizer.transform([processed_text])

predicted_label = model.predict(vectorized_text)[0]
print("Predicted Label:", predicted_label)
# Decode to string label
predicted_sentiment = label_encoder.inverse_transform([predicted_label])[0]
print("Sentiment:", predicted_sentiment)

Predicted Label: 1
Sentiment: Negative


In [17]:
# Save the model and vectorizer
import pickle

pickle.dump(model, open("src/sentiment_model.pkl", "wb"))
pickle.dump(vectorizer, open("src/vectorizer.pkl", "wb"))